
# 03b — Re-correct GSVA stats (per cell type)

Switches the BH correction from pooled-across-all-cell-types to **per cell type ×
comparison**, matching how the FGSEA p-values were corrected (each cell type
corrected across its own pathways). Operates on the raw `p_value` already saved in
the stats tables, so no tests are rerun. Rewrites both files in place.

In [1]:
suppressPackageStartupMessages({
  library(data.table)
  library(dplyr)
})

signif_cut <- function(p) {
  cut(p,
    breaks = c(-Inf, 1e-4, 1e-3, 1e-2, 5e-2, Inf),
    labels = c("****", "***", "**", "*", "ns")
  )
}

for (t in c("pbmc", "bmmc")) {
  f <- sprintf("../../../data/rna/gsva/results/gsva_results/gsva_stats_%s.csv", t)
  adj <- sprintf("../../../data/rna/gsva/results/gsva_results/gsva_stats_%s_adjusted.csv", t)

  fread(f) %>%
    group_by(celltype, source = sub("_.*", "", pathway), group1, group2) %>% # HALLMARK / REACTOME / ENCODE / KEGG
    mutate(p_adj_BH = p.adjust(p_value, method = "BH")) %>%
    ungroup() %>%
    select(-source) %>%
    mutate(p_signif = signif_cut(p_adj_BH))

  message(t, ": wrote ", adj)
}

pbmc: wrote results/gsva_results/gsva_stats_pbmc_adjusted.csv

bmmc: wrote results/gsva_results/gsva_stats_bmmc_adjusted.csv



In [3]:
for (t in c("pbmc", "bmmc")) {
  for (suf in c("", "_adjusted")) {
    f <- sprintf("../../../data/rna/gsva/results/gsva_results/gsva_stats_%s%s.csv", t, suf)
    cat("\n===", t, if (suf == "") "(original)" else "(adjusted)", "===\n")
    print(fread(f) %>% filter(p_signif != "ns") %>% count(test))
  }
}


=== pbmc (original) ===
            test     n
          <char> <int>
1:     limma_adj 41338
2: wilcox_paired  1155

=== pbmc (adjusted) ===
            test     n
          <char> <int>
1:     limma_adj 47279
2: wilcox_paired  7823

=== bmmc (original) ===
        test     n
      <char> <int>
1: limma_adj  8299

=== bmmc (adjusted) ===
            test     n
          <char> <int>
1:     limma_adj 11973
2: wilcox_paired   443


Per-sample pathway activity (GSVA). To assess pathway activity at the level of individual samples, we computed gene set variation analysis (GSVA) scores [Hänzelmann et al., 2013] from the same cell-type pseudobulk profiles used for differential expression. For each cell type, genes were retained if detected in ≥10% of cells, excluding mitochondrial, ribosomal, hemoglobin, and immunoglobulin genes, and counts were variance-stabilized with DESeq2 [Love et al., 2014]. GSVA scores were calculated against gene sets from MSigDB, Hallmark 2020, Reactome 2024, ENCODE/ChEA. Consensus TFs, and KEGG 2021 Human (Enrichr collections), restricted to sets of 10–500 genes. Scores were summarized to one value per subject and visit. Longitudinal comparisons between visits (same subjects) were tested with paired Wilcoxon signed-rank tests, and comparisons against healthy donors (independent subjects) were tested with linear models (limma [Ritchie et al., 2015]) adjusting for sex, age, and CMV serostatus, matching the covariates used in the pseudobulk differential expression analysis. P-values were corrected for multiple testing using the Benjamini–Hochberg method within each cell type and pathway database, consistent with the correction applied in the FGSEA analysis of the primary figures.

## Edit for data apps

In [ ]:
library(arrow)        # install.packages("arrow") if needed
library(data.table)
library(dplyr)

od <- "../../../data/rna/gsva/results/gsva_results"

for (t in c("pbmc", "bmmc")) {
  stats <- fread(file.path(od, sprintf("gsva_stats_%s_adjusted.csv", t))) %>%
    mutate(
      foreground = group2,                       # test group (effect = group2 - group1)
      background = group1,                        # reference group
      contrast   = paste0(group2, "_vs_", group1)
    )
  write_parquet(stats, file.path(od, sprintf("gsva_stats_%s_dataapps.parquet", t)))

  scores <- fread(file.path(od, sprintf("gsva_scores_%s.csv", t)))
  write_parquet(scores, file.path(od, sprintf("gsva_scores_%s_dataapps.parquet", t)))

  message("wrote _dataapps parquet for ", t)
}